# NB2b-post — Entity Prefix Merge (fast, no GPU)

A small cleanup pass over my `extract_algo.parquet`. My NER run is already good (0%
fragmentation after the `first` + word-aligned chunking fixes), but Arabic attaches
preposition/conjunction prefixes (بـ, لـ, و, ال...) to entities, so the **same** entity can
appear as several forms — e.g. حماس / وحماس / بحماس — inflating the entity count and
distorting later coverage checks.

**My rule (safe):** I only strip a prefix when the stripped form already exists in the same
list (proof it's a real prefix). So "حماس" absorbs "وحماس"/"بحماس", but a standalone name like
**"بغداد" is never touched** (there's no "غداد" to match). Forms that appear only once with a
prefix (e.g. "والجهاد الإسلامي") I leave as-is here — those are handled later by fuzzy matching
at the acceptance gate, not by risky stripping.

**No GPU needed** — this is pure string post-processing on the existing file.

In [1]:
import pandas as pd
import json
import re

IN_PATH  = '/kaggle/input/notebooks/bahaaqassem/nb2b-algorithmic-extraction/extract_algo.parquet'   # my NER output
OUT_DIR  = '/kaggle/working'

df = pd.read_parquet(IN_PATH)
print('shape:', df.shape)
print('columns:', df.columns.tolist())

shape: (3500, 10)
columns: ['id', 'pair_id', 'entities', 'numbers_dates', 'source_agencies', 'key_sentences', 'quote_events', 'n_fact_points', 'target_words', 'error']


In [2]:
def strip_prefix_candidate(e):
    # The candidate form after removing ONE leading prefix (for comparison/merge only)
    m = re.match(r'^[وفبك](ال.+)', e)          # وال/بال/فال/كال -> ال
    if m: return m.group(1)
    if e.startswith('لل') and len(e) > 3:        # لل -> ال
        return 'ال' + e[2:]
    m = re.match(r'^[وفبكل](.{3,})', e)          # single جر/عطف letter + word(>=3) -> word
    if m: return m.group(1)
    return None

def root_form(e):
    # Strip prefixes iteratively to reach the shortest form (the merge key)
    key = e; c = strip_prefix_candidate(e); seen = {e}
    while c and c not in seen:
        seen.add(c); key = c; c = strip_prefix_candidate(key)
    return key

def dedupe_prefixed(items):
    # Merge prefix variants safely; keep the cleanest form; never distort standalone names
    if not items:
        return items
    items_set = set(items)
    # 1) drop e if its stripped form is explicitly present in the list
    kept = [e for e in items if strip_prefix_candidate(e) not in items_set]
    # 2) group the rest by root and keep the cleanest form per group
    groups = {}
    for e in kept:
        groups.setdefault(root_form(e), []).append(e)
    final = []
    for key, forms in groups.items():
        best = min(forms, key=lambda x: (x != key, len(x)))   # equals-key first, then shortest
        final.append(best)
    return final

def clean_entities_json(ej):
    ent = json.loads(ej)
    return json.dumps({k: dedupe_prefixed(v) for k, v in ent.items()}, ensure_ascii=False)

## Apply and report the effect

In [3]:
before_total = df['entities'].apply(lambda x: sum(len(v) for v in json.loads(x).values())).sum()

df['entities'] = df['entities'].apply(clean_entities_json)   # overwrite in place

after_total = df['entities'].apply(lambda x: sum(len(v) for v in json.loads(x).values())).sum()

print('entities before:', f'{before_total:,}')
print('entities after: ', f'{after_total:,}')
print('merged (dup prefixes removed):', f'{before_total-after_total:,}',
      f'({100*(before_total-after_total)/before_total:.1f}%)')
print('avg entities/article:', round(before_total/len(df),1), '->', round(after_total/len(df),1))

# spot check
import json as _j
for i in range(2):
    print(f"\n=== {df.iloc[i]['id']} ===")
    print('  orgs:', _j.loads(df.iloc[i]['entities'])['organizations'])
    print('  persons:', _j.loads(df.iloc[i]['entities'])['persons'])

entities before: 79,887
entities after:  72,128
merged (dup prefixes removed): 7,759 (9.7%)
avg entities/article: 22.8 -> 20.6

=== HA_00000 ===
  orgs: ['حلف شمال الأطلسي', 'يديعوت أحرونوت', 'الغارديان', 'الموساد']
  persons: ['رجب طيب أردوغان', 'صياء عبد الرضا', 'جورج دبليو بوش', 'دافيد بارنياع', 'أنتوني بلينكن', 'دافيد برنياع', 'نفتالي بينيت', 'موشيه يعلون', 'بيني غانتس']

=== HA_00001 ===
  orgs: ['جبهة النصرة', 'للجزيرة نت', 'إنترفاكس']
  persons: ['وائل الحلقي', 'أديب عليوي', 'محمد خطاب']


## Safety check — standalone names must NOT be altered

In [4]:
for name in ['بغداد','بيروت','لبنان','فلسطين','كتائب','وليد','غزة','رفح']:
    print(f'  {name} -> {dedupe_prefixed([name])}')

  بغداد -> ['بغداد']
  بيروت -> ['بيروت']
  لبنان -> ['لبنان']
  فلسطين -> ['فلسطين']
  كتائب -> ['كتائب']
  وليد -> ['وليد']
  غزة -> ['غزة']
  رفح -> ['رفح']


In [5]:
# Save the cleaned file (same schema, entities column merged)
df.to_parquet(f'{OUT_DIR}/extract_algo.parquet', index=False)
print('saved:', df.shape)

saved: (3500, 10)


## Notes

- I overwrite the `entities` column so the output keeps the exact same schema as NB2b —
  downstream NB2c reads it unchanged.
- Remaining single-occurrence prefixed forms (e.g. "والجهاد الإسلامي") are intentionally kept;
  I resolve those with fuzzy matching at the acceptance gate rather than risk distorting a real
  name here.
- I upload this as `aigt-extract-algo` (overwriting the previous version) so NB2c uses the
  cleaned entities.